# DocGuard-VLM: LoRA fine-tuning of Qwen2-VL-2B for document OCR + forgery detection

Runs on a free Colab **T4** GPU via [unsloth](https://github.com/unslothai/unsloth) QLoRA.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then confirm with `!nvidia-smi`
in a scratch cell (should list a Tesla T4) before running anything else.

**Note on Colab disconnects:** free-tier Colab can reclaim/reset your VM (idle timeout or
otherwise), which wipes the whole disk, not just the Python kernel. Training checkpoints are
saved to Google Drive (see the Drive-mount cell below) specifically so a disconnect mid-training
doesn't lose progress. If it happens: `Runtime -> Run all` and training will resume from the
last saved checkpoint automatically instead of restarting from scratch.

Just run every cell below in order, top to bottom (or `Runtime -> Run all`). The first code cell
clones the repo and regenerates the dataset (`data/processed/` is ~1.5GB of images and isn't
committed to git, so it's rebuilt here instead of uploaded) — takes a couple of minutes.

In [ ]:
import os

REPO_URL = "https://github.com/kratos0718/docguard-vlm.git"

if not os.path.exists("/content/docguard-vlm"):
    !git clone {REPO_URL} /content/docguard-vlm
%cd /content/docguard-vlm
!pip install -q datasets huggingface_hub tqdm pillow opencv-python-headless
!PYTHONPATH=src python src/data_gen/build_dataset.py --out-dir data/processed \
    --n-train 350 --n-test-clean 60 --n-test-adv 60

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Training checkpoints/adapter get saved here instead of Colab's ephemeral disk, so a
# disconnect/VM-reset mid-training doesn't lose progress. The dataset itself (data/processed/)
# stays on local disk -- it's cheap to rebuild (a few minutes) and not worth the extra
# complexity of persisting 1.5GB of images to Drive.
import os
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/docguard-vlm-outputs/qwen2vl-2b-docguard-lora"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("checkpoints will be saved to:", DRIVE_OUTPUT_DIR)

In [ ]:
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install -q trl peft accelerate bitsandbytes pillow
# Note: don't pin trl to an old version here -- unsloth/peft/transformers move fast and an
# outdated trl pin can drift out of sync with a freshly-installed peft, causing NameErrors
# like `VARIANT_KWARG_KEYS` inside unsloth's auto-generated compiled kernels. If you hit that
# error: `!rm -rf unsloth_compiled_cache`, upgrade unsloth+unsloth_zoo, then Runtime > Restart
# session before re-running (pip upgrades don't reload already-imported modules).

In [ ]:
DATA_ROOT = "data/processed"  # adjust if you cloned/uploaded elsewhere
OUTPUT_DIR = DRIVE_OUTPUT_DIR  # persists across disconnects; see the Drive-mount cell above
BASE_MODEL = "unsloth/Qwen2-VL-2B-Instruct"
MAX_STEPS = None       # set an int to cap steps for a quick run; None = full epochs
NUM_EPOCHS = 2
LORA_R = 16
LORA_ALPHA = 16

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

## Load DocGuard-VLM JSONL and convert to unsloth's vision chat format

Each record already stores an instruction/response pair (OCR field-extraction or forgery-verdict) built by `src/data_gen/build_dataset.py`.

In [ ]:
import json, os
from PIL import Image

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def to_conversation_sample(record, data_root):
    img = Image.open(os.path.join(data_root, record["image"])).convert("RGB")
    messages = []
    for turn in record["conversations"]:
        content = []
        for c in turn["content"]:
            if c["type"] == "image":
                content.append({"type": "image", "image": img})
            else:
                content.append({"type": "text", "text": c["text"]})
        messages.append({"role": turn["role"], "content": content})
    return {"messages": messages}

train_records = load_jsonl(os.path.join(DATA_ROOT, "train.jsonl"))
train_dataset = [to_conversation_sample(r, DATA_ROOT) for r in train_records]
print(f"train samples: {len(train_dataset)}")
print(train_dataset[0]["messages"][0]["content"][1]["text"])
print(train_dataset[0]["messages"][1]["content"][0]["text"])

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import glob

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS if MAX_STEPS else -1,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        # vision-specific: unsloth's collator needs these off so it can pack image+text itself
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
    ),
)

# OUTPUT_DIR lives on Drive, so if a previous run got disconnected mid-training, its
# checkpoint-* folders are still there -- resume from the latest one instead of restarting.
existing_checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
resume_from = existing_checkpoints[-1] if existing_checkpoints else None
if resume_from:
    print(f"resuming from checkpoint: {resume_from}")
trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Save the LoRA adapter (small, a few hundred MB) -- this is what you download
# and hand to src/eval/evaluate.py for scoring.
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved to", OUTPUT_DIR)

## Quick smoke test
Run one clean and one adversarial example through the fine-tuned model before doing the full evaluation harness.

In [ ]:
FastVisionModel.for_inference(model)

test_records = load_jsonl(os.path.join(DATA_ROOT, "test_clean.jsonl"))
sample = next(r for r in test_records if r["task"] == "forgery")
img = Image.open(os.path.join(DATA_ROOT, sample["image"])).convert("RGB")
instruction = sample["conversations"][0]["content"][1]["text"]

messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(img, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

out = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.2)
print("GROUND TRUTH:", sample["conversations"][1]["content"][0]["text"])
print("PREDICTION  :", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Next: run the full evaluation harness
The adapter is saved under `OUTPUT_DIR` (on Google Drive: `/content/drive/MyDrive/docguard-vlm-outputs/qwen2vl-2b-docguard-lora`).
Download it, or continue in this notebook:
```
!python src/eval/evaluate.py \
  --data-root data/processed \
  --adapter "/content/drive/MyDrive/docguard-vlm-outputs/qwen2vl-2b-docguard-lora" \
  --base-model unsloth/Qwen2-VL-2B-Instruct \
  --out results/eval_results.json
```
This scores zero-shot baseline vs. fine-tuned, on both `test_clean.jsonl` and `test_adversarial.jsonl`.